### References

https://github.com/gabrielziegler3/xgboost-multiclass-multilabel/blob/master/xgboost-multiclass/multiclass-frog-classification.ipynb


In [4]:
import os

import mlflow
import mlflow.xgboost
import mlflow.sklearn

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display
from sklearn.feature_selection import mutual_info_regression
import plotly

from sklearn.model_selection import train_test_split
from xgboost.sklearn import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import recall_score, f1_score, precision_score, precision_recall_fscore_support
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.datasets import make_classification

import matplotlib.pyplot as plt
from matplotlib import pyplot

In [5]:
not_use_cols = ['date']
attr_cols = ['open', 'high', 'low', 'close', 'volume', 'average', 'barCount']
h1s = ['h1s_high_max', 'h1s_low_min', 'h1s_barCount_sum', 'h1s_volume_sum', 'h1s_average_avg' ]
h2s = ['h2s_high_max', 'h2s_low_min', 'h2s_barCount_sum', 'h2s_volume_sum', 'h2s_average_avg' ]
h3s = ['h3s_high_max', 'h3s_low_min', 'h3s_barCount_sum', 'h3s_volume_sum', 'h3s_average_avg' ]
h4s = ['h4s_high_max', 'h4s_low_min', 'h4s_barCount_sum', 'h4s_volume_sum', 'h4s_average_avg' ]
hist_5_cols = ['h5s_high_max',  'h5s_low_min',  'h5s_barCount_sum',  'h5s_volume_sum', 'h5s_average_avg']
hist_10_cols= ['h10s_high_max', 'h10s_low_min', 'h10s_barCount_sum', 'h10s_volume_sum','h10s_average_avg']
hist_15_cols= ['h15s_high_max', 'h15s_low_min', 'h15s_barCount_sum', 'h15s_volume_sum','h15s_average_avg']
pred_cols= [ 'f5s_average', 'f5s_10c_arrow', 'f5s_15c_arrow', 'f5s_20c_arrow',  'f5s_25c_arrow' ]
X_cols = [ ]
y_cols = [ ]
X_cols += attr_cols + h1s + h2s + h3s +  h4s
x_cols  = [ 'barCount',          'volume',        'average',
            'h1s_barCount_sum', 'h1s_volume_sum', 'h1s_average_avg' ,
            'h2s_barCount_sum', 'h2s_volume_sum', 'h2s_average_avg' ,
            'h3s_barCount_sum', 'h3s_volume_sum', 'h3s_average_avg',
            'h4s_barCount_sum', 'h4s_volume_sum', 'h4s_average_avg'  ]
y_cols += pred_cols

bin_counts = 3
data_file = "./contract-TSLA/All.csv"


## mlflow logging

### [reference page](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-log-view-metrics?tabs=interactive)

In [6]:
mlflow.set_experiment("vol-model-knn")
mlflow.xgboost.autolog()

In [7]:
plt.style.use("seaborn-whitegrid")
plt.rc("figure", autolayout=True)
plt.rc(
    "axes",
    labelweight="bold",
    labelsize="large",
    titleweight="bold",
    titlesize=14,
    titlepad=10,
)

In [3]:
def load_data():
    _data = pd.read_csv(data_file, low_memory=False)
    _data = _data.dropna()
    print ("size", _data.shape)
    mlflow.log_param("Size", _data.shape)
    mlflow.log_param("X cols", X_cols)
    mlflow.log_param("y col",  y_cols[1:])
    _data = _data.head(1000000)
    return _data

# Code

## load data / split data

In [ ]:
%%time
data = load_data()
X = data[X_cols]
y = data [y_cols [1:]]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
msg = f"Train {X_train.shape},{y_train.shape}, Test {X_test.shape}, {y_test.shape}"
print (msg)

## Scale Data

In [ ]:
# Standard Scaled X Prediction
scalar = StandardScaler()
scalar.fit(X_train)
X_train = pd.DataFrame(scalar.transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(scalar.transform(X_test), columns=X_test.columns)
#x_train = scalar.transform(x_train)
#x_test = scalar.transform(x_test)


## Create KNN Model

In [ ]:
%%time
#Y_cols = [ 'f5s_average', 'f5s_10c_arrow', 'f5s_15c_arrow', 'f5s_20c_arrow',  'f5s_25c_arrow']
# https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html

from sklearn.neighbors import  KNeighborsClassifier
knn_f10 = KNeighborsClassifier(n_neighbors=3, n_jobs=5)
knn_f15 = KNeighborsClassifier(n_neighbors=3, n_jobs=5)
knn_f20 = KNeighborsClassifier(n_neighbors=3, n_jobs=5)
knn_f25 = KNeighborsClassifier(n_neighbors=3, n_jobs=5)

print(knn_f10.get_params())
#mlflow.log_param('knn_params', knn.get_xgb_params())
#mlflow.knn.autolog()


In [ ]:
y_train.columns

In [ ]:
%%time
# Train model & predict
knn_f10.fit(X_train, y_train['f5s_10c_arrow'])
knn_f15.fit(X_train, y_train['f5s_15c_arrow'])
knn_f20.fit(X_train, y_train['f5s_20c_arrow'])
knn_f25.fit(X_train, y_train['f5s_25c_arrow'])

In [ ]:
%%time
y_f10_pred = knn_f10.predict(X_test)
y_f15_pred = knn_f15.predict(X_test)
y_f20_pred = knn_f20.predict(X_test)
y_f25_pred = knn_f25.predict(X_test)

In [ ]:
%%time
#average setting, one of [None, 'micro', 'macro', 'weighted']
# precision, recall, f1, y_true = precision_recall_fscore_support(y_test, y_pred, average='weighted')

ps = precision_score(y_test['f5s_10c_arrow'], y_f10_pred, average='weighted')
rs = recall_score(y_test['f5s_10c_arrow'], y_f10_pred, average='weighted')
ac = accuracy_score(y_test['f5s_10c_arrow'], y_f10_pred)
print(f"f10_accuracy: {round(ac, 3)} f10_precision {round(ps, 3)} f10_recall {round(rs, 3)}")
mlflow.log_metrics({'f10_accuracy': round(ac, 3), 'f10_precision': round(ps, 3), 'f10_recall': round(rs, 3)})


ps = precision_score(y_test['f5s_15c_arrow'], y_f15_pred, average='weighted')
rs = recall_score(y_test['f5s_15c_arrow'], y_f15_pred, average='weighted')
ac = accuracy_score(y_test['f5s_15c_arrow'], y_f15_pred)
print(f"f15_accuracy: {round(ac, 3)} f15_precision {round(ps, 3)} f15_recall {round(rs, 3)}")
mlflow.log_metrics({'f15_accuracy': round(ac, 3), 'f15_precision': round(ps, 3), 'f15_recall': round(rs, 3)})

ps = precision_score(y_test['f5s_20c_arrow'], y_f20_pred, average='weighted')
rs = recall_score(y_test['f5s_20c_arrow'], y_f20_pred, average='weighted')
ac = accuracy_score(y_test['f5s_20c_arrow'], y_f20_pred)
print(f"f20_accuracy: {round(ac, 3)} f20_precision {round(ps, 3)} f20_recall {round(rs, 3)}")
mlflow.log_metrics({'f20_accuracy': round(ac, 3), 'f20_precision': round(ps, 3), 'f20_recall': round(rs, 3)})

ps = precision_score(y_test['f5s_25c_arrow'], y_f25_pred, average='weighted')
rs = recall_score(y_test['f5s_25c_arrow'], y_f25_pred, average='weighted')
ac = accuracy_score(y_test['f5s_25c_arrow'], y_f25_pred)
print(f"f25_accuracy: {round(ac, 3)} f25_precision {round(ps, 3)} f25_recall {round(rs, 3)}")
mlflow.log_metrics({'f25_accuracy': round(ac, 3), 'f25_precision': round(ps, 3), 'f25_recall': round(rs, 3)})


In [ ]:
def plot_confusion_matrix(cm, classes, normalized=True, cmap='bone', fmt='.3g'):
    #plt.figure(figsize=[7, 6])
    plt.figure(figsize=[14, 12])
    norm_cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    print(cm)
    print(norm_cm)

    arr = [["-"]*len(cm)]*len(cm)
    for j in range(len(cm)):
        for i in range(len(cm[j])):
            arr[j][i] =  f"{cm[j][i]:,d}\n{norm_cm[j][i]:.3f}%"
            print(cm[j][i], arr[j][i])
    print(arr)
    sns.heatmap(norm_cm, annot=arr, annot_kws={'size': 15}, fmt='s', normalize=True)

In [ ]:
def plot_confusion_matrix2(_cm, _title):
    plt.figure(figsize=[14, 12])
    disp = ConfusionMatrixDisplay(confusion_matrix=_cm, display_labels=[-1, 0, 1])
    disp.plot(cmap=plt.cm.Paired,  values_format=',')
    fig = disp.ax_.get_figure()
    fig.set_figwidth(12)
    fig.set_figheight(12)
    # print (plt.rcParams.keys())
    plt.rcParams.update({'font.size': 22})
    plt.show()
    mlflow.log_figure(fig, _title)


In [ ]:
cm_f10 = confusion_matrix(y_test['f5s_10c_arrow'], y_f10_pred, labels=[-1,0,1])
cm_f15 = confusion_matrix(y_test['f5s_15c_arrow'], y_f15_pred, labels=[-1,0,1])
cm_f20 = confusion_matrix(y_test['f5s_20c_arrow'], y_f20_pred, labels=[-1,0,1])
cm_f25 = confusion_matrix(y_test['f5s_25c_arrow'], y_f25_pred, labels=[-1,0,1])
# norm_cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
plot_confusion_matrix2(cm_f10, 'f10_confusion_matrix.png')
plot_confusion_matrix2(cm_f15, 'f15_confusion_matrix.png')
plot_confusion_matrix2(cm_f20, 'f20_confusion_matrix.png')
plot_confusion_matrix2(cm_f25, 'f25_confusion_matrix.png')

print(cm_f10)
print(cm_f15)
print(cm_f20)
print(cm_f25)


In [ ]:
print(cm_f10)
print(cm_f15)
print(cm_f20)
print(cm_f25)

In [ ]:
cm_f10_json = {}
cm_f10_json.update({"f10_Confusion_Matrix": cm_f10})
mlflow.log_param("f10_confusion_matrix.json", cm_f10_json)
print(cm_f10_json)

cm_f15_json = {}
cm_f15_json.update({"f15_Confusion_Matrix": cm_f15})
mlflow.log_param("f15_confusion_matrix.json", cm_f15_json)
print(cm_f15_json)

cm_f20_json = {}
cm_f20_json.update({"f20_Confusion_Matrix": cm_f20})
mlflow.log_param("f20_confusion_matrix.json", cm_f20_json)
print(cm_f20_json)

cm_f25_json = {}
cm_f25_json.update({"f25_Confusion_Matrix": cm_f25})
mlflow.log_param("f25_confusion_matrix.json", cm_f25_json)
print(cm_f25_json)

In [ ]:
# from mlflow.models.signature import infer_signature
# signature = infer_signature(x_train, xgb.predict(xgb.DMatrix(data=x_train, label=y_train)))
# mlflow.xgboost.log_model(xgb, "model") #, signature=signature)""
#fig, ax = plt.subplots()
#sns.set(rc = {'figure.figsize':(12,9)})
#ns.heatmap(cm, annot=True, cmap="YlGnBu",fmt='d')
#fig.savefig("ConfusionMatrix.png")
#mlflow.log_figure(fig, "ConfusionMatrix.png")
#plt.show()

#sns.heatmap(cm_pct, annot=True, cmap="YlGnBu", fmt=".3f")
#fig.savefig("ConfusionMatrixPct.png")
#mlflow.log_figure(fig, "ConfusionMatrixPct.png")
#plt.show()


# feature_importances = pd.DataFrame(xgb.feature_importances_,index=x_train.columns.tolist(),columns=['importance'])
# feature_importances.sort_values('importance', ascending=False)


In [ ]:
def print_fi(xgb, name):
    feature_importance = pd.DataFrame(xgb.feature_importances_,index=X_train.columns.tolist(),columns=['importance'])
    vals = feature_importance.sort_values('importance', ascending=False)
    mlflow.log_param(name, vals.to_string)
    print (name)
    print(vals)


    #_x = pd.Series(xgb.feature_importances_, index=X_train.columns).nlargest(12).plot(kind='barh')
    _x = pyplot.barh(range(len(xgb.feature_importances_)), xgb.feature_importances_, )
    plt.show()
    #mlflow.log_image(image, name)
    return

In [ ]:
print_fi( xgb_f10, 'f10_feature_importance')
print_fi( xgb_f15, 'f15_feature_importance')
print_fi( xgb_f20, 'f20_feature_importance')
print_fi( xgb_f25, 'f25_feature_importance')

In [ ]:
mlflow.end_run()

# ToDo

* add history: h1, h2, h3, h4, h5.  There seems to be no value for >5
* filter time to 9:45 - 15:45
* remove non-full days
* add vix index
* move data to MySQL

DONE:

* Create multiple arrow buckets ('f5s_10c_arrow', 'f5s_20c_arrow') += 'f5s_15c_arrow', 'f5s_25c_arrow', 'f5s_30c_arrow'

In [ ]:
print(X_test)

In [ ]:
output = X_test.copy()

In [ ]:
cm_f10 = confusion_matrix(y_test['f5s_10c_arrow'], y_f10_pred, labels=[-1,0,1])
cm_f15 = confusion_matrix(y_test['f5s_15c_arrow'], y_f15_pred, labels=[-1,0,1])
cm_f20 = confusion_matrix(y_test['f5s_20c_arrow'], y_f20_pred, labels=[-1,0,1])
cm_f25 = confusion_matrix(y_test['f5s_25c_arrow'], y_f25_pred, labels=[-1,0,1])

In [ ]:
print (y_test)
y_test['p_f5s_10c_arrow'] = y_f10_pred
y_test['p_f5s_15c_arrow'] = y_f15_pred
y_test['p_f5s_20c_arrow'] = y_f20_pred
y_test['p_f5s_25c_arrow'] = y_f25_pred
print (y_test)

In [ ]:
print(X_test.shape)
print(y_test.shape)

to_save = X_test.join(y_test)
to_save.head(100)

#to_save = X_test.copy()
#to_save[ y_test.columns ] = y_test [ y_test.columns ]
print(to_save.shape)

In [ ]:
print (y_test.shape)
print (X_test.shape)
print (to_save.shape)

In [ ]:
to_save.head()

In [ ]:
to_save.to_csv("test_pred.csv")

In [ ]:
y_test.head()

In [ ]:
X_test.head()